# 03 Tree Models

Use this notebook to compare stronger nonlinear models against the logistic regression baseline.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.fraud_detection.data import load_train_data, split_features_target
from src.fraud_detection.features import build_preprocessor, drop_high_missing_columns
from src.fraud_detection.metrics import compute_classification_metrics

In [ ]:
data = load_train_data(sample_size=50000)
X, y = split_features_target(data)
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train, dropped_columns = drop_high_missing_columns(X_train, threshold=0.95)
X_valid = X_valid.drop(columns=dropped_columns, errors="ignore")
preprocessor, _, _ = build_preprocessor(X_train)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

model.fit(X_train, y_train)
valid_scores = model.predict_proba(X_valid)[:, 1]
pd.Series(compute_classification_metrics(y_valid, valid_scores))

## Next Extensions

- Replace `RandomForestClassifier` with `XGBClassifier` or `LGBMClassifier`.
- Tune depth, learning rate, number of estimators, and class weighting.
- Compare ROC-AUC and PR-AUC against the baseline logistic model.